In [7]:
# xorkit_dynamic_numeric_fixed.py
import os
import hmac
import hashlib
from typing import Dict, Any

# ---------------- Configuration ----------------
PBKDF2_ITERS = 200_000
MASTER_KEY_LEN = 32            # 256-bit master key
FLUID_NUM_DIGITS = 12         # must be multiple of 3 (e.g., 12 -> 4 bytes per char)
DIVISION_DIGITS = 3           # must be 3 so each mapped digit -> one byte (0..255)
KEYOP_DIGITS = 6
CHARSET = [chr(i) for i in range(32, 127)]  # printable ASCII
# ------------------------------------------------

# ---------- KDF / Subkey Derivation ----------
def derive_master_key(password: str, salt: bytes) -> bytes:
    """Derive a fixed-length master key from password+salt using PBKDF2-HMAC-SHA256."""
    return hashlib.pbkdf2_hmac('sha256', password.encode('utf-8'), salt, PBKDF2_ITERS, dklen=MASTER_KEY_LEN)

def derive_subkey(master_key: bytes, info: bytes) -> bytes:
    """Derive deterministic subkey using HMAC-SHA256(master_key, info)."""
    return hmac.new(master_key, info, hashlib.sha256).digest()

def drbg_bytes(subkey: bytes, label: bytes, counter: int, out_bytes: int) -> bytes:
    """Deterministic DRBG-like expansion via HMAC chaining; returns `out_bytes` bytes."""
    out = bytearray()
    ctr = counter
    while len(out) < out_bytes:
        out.extend(hmac.new(subkey, label + ctr.to_bytes(4, 'big'), hashlib.sha256).digest())
        ctr += 1
    return bytes(out[:out_bytes])

# Produce decimal string of length `digits` from HMAC-derived bytes (not used for byte-group tables)
def drbg_chunk(subkey: bytes, label: bytes, counter: int, digits: int) -> str:
    msg = label + counter.to_bytes(4, 'big')
    out = hmac.new(subkey, msg, hashlib.sha256).digest()
    val = int.from_bytes(out, 'big') % (10 ** digits)
    return str(val).zfill(digits)

# ---------- Table Generators ----------
def gen_fluid_table(subkey: bytes, charset=CHARSET, digits=FLUID_NUM_DIGITS) -> Dict[str, str]:
    """
    Deterministic Fluid table: produce per-character numeric string of length `digits`.
    Implementation: generate `digits//3` groups of 3-digit numbers each in [000..255] and concatenate.
    """
    if digits % 3 != 0:
        raise ValueError("FLUID_NUM_DIGITS must be a multiple of 3.")
    groups = digits // 3
    table = {}
    for i, ch in enumerate(charset):
        parts = []
        for g in range(groups):
            # produce 2 bytes of HMAC-derived material and map to 0..255 deterministically
            # use drbg_bytes to get at least 2 bytes, then interpret and mod 256
            raw = drbg_bytes(subkey, b'fluid_' + bytes([i % 256]) + bytes([g % 256]), counter=g, out_bytes=2)
            val = int.from_bytes(raw, 'big') % 256
            parts.append(f"{val:03d}")
        table[ch] = "".join(parts)
    return table

def gen_number_division_table(subkey: bytes, digits=DIVISION_DIGITS) -> Dict[str, str]:
    """
    Deterministic Number division table: digit '0'..'9' -> decimal string of length `digits`.
    For digits==3 we ensure values are in 0..255 so each maps to exactly one byte.
    """
    if digits != 3:
        # support only 3 for the current design
        raise ValueError("DIVISION_DIGITS must be 3 in this implementation.")
    table = {}
    for d in range(10):
        raw = drbg_bytes(subkey, b'div_' + bytes([d]), counter=d, out_bytes=2)
        val = int.from_bytes(raw, 'big') % 256
        table[str(d)] = f"{val:03d}"
    return table

# ---------- Byte-oriented keystream ----------
def gen_xor_mask_bytes(subkey: bytes, length_bytes: int) -> bytes:
    """Deterministic pseudorandom byte stream from subkey (HMAC chaining)."""
    out = bytearray()
    counter = 0
    while len(out) < length_bytes:
        blk = hmac.new(subkey, b'xor_bytes_' + counter.to_bytes(4, 'big'), hashlib.sha256).digest()
        out.extend(blk)
        counter += 1
    return bytes(out[:length_bytes])

# ---------- Decimal <-> bytes (bijective) ----------
def digits_to_bytes(numeric_string: str) -> bytes:
    """
    Convert decimal string whose length is multiple of 3 into bytes:
    each 3-digit group (000..255) -> integer byte 0..255.
    """
    assert len(numeric_string) % 3 == 0, "numeric_string length must be multiple of 3"
    b = bytearray()
    for i in range(0, len(numeric_string), 3):
        chunk = numeric_string[i:i+3]
        val = int(chunk)
        if val > 255:
            # This should not happen because generation clamps to 0..255
            val = val % 256
        b.append(val)
    return bytes(b)

def bytes_to_digits(b: bytes) -> str:
    """Convert bytes -> decimal string with 3-digit zero-padded groups."""
    return ''.join(f'{x:03d}' for x in b)

# ---------- XOR helper ----------
def xor_bytes(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))

# ---------- Main encrypt / decrypt ----------
def encrypt_numeric(password: str, plaintext: str) -> Dict[str, Any]:
    """
    Encrypt plaintext (string).
    Returns envelope: {salt, cipher (numeric string), tags, meta}
    """
    salt = os.urandom(16)
    master = derive_master_key(password, salt)

    # Stage subkeys
    sk1 = derive_subkey(master, b"stage1-fluid")
    sk2 = derive_subkey(master, b"stage2-div")
    sk3 = derive_subkey(master, b"stage3-keyop")
    sk4 = derive_subkey(master, b"stage4-xor")

    # Tables (deterministic)
    Fluid_table = gen_fluid_table(sk1, charset=CHARSET, digits=FLUID_NUM_DIGITS)   # char -> FLUID_NUM_DIGITS digits
    Number_div = gen_number_division_table(sk2, digits=DIVISION_DIGITS)           # digit -> 3-digit code

    # Stage 1: characters -> fluid numeric string
    concat = ""
    for ch in plaintext:
        if ch not in Fluid_table:
            raise ValueError(f"Character not supported in charset: {repr(ch)}")
        concat += Fluid_table[ch]   # length = len(plaintext) * FLUID_NUM_DIGITS

    # Stage 2: digit -> 3-digit division chunk (concatenate)
    stage2 = "".join(Number_div[d] for d in concat)   # length multiple of 3

    # Convert stage2 decimal string -> bytes (3 digits -> 1 byte)
    stage2_bytes = digits_to_bytes(stage2)

    # Stage 3: chunk-wise XOR with sk3-derived per-chunk keystream
    # chunk size in bytes: DIVISION_DIGITS/3 => 1
    chunk_bytes = DIVISION_DIGITS // 3
    chunks = [stage2_bytes[i:i+chunk_bytes] for i in range(0, len(stage2_bytes), chunk_bytes)]
    stage3_parts = []
    for idx, ch_bytes in enumerate(chunks):
        ks = gen_xor_mask_bytes(hmac.new(sk3, b'chunkidx_' + idx.to_bytes(4,'big'), hashlib.sha256).digest(), len(ch_bytes))
        token = xor_bytes(ch_bytes, ks)
        stage3_parts.append(token)
    stage3_all_bytes = b''.join(stage3_parts)

    # Stage 4: XOR whole stream with sk4 keystream
    ks_full = gen_xor_mask_bytes(sk4, len(stage3_all_bytes))
    cipher_bytes = xor_bytes(stage3_all_bytes, ks_full)

    # Convert bytes -> numeric string (3-digit per byte)
    cipher_digits = bytes_to_digits(cipher_bytes)

    # Stage verification tags (HMACs)
    tag1 = hmac.new(sk1, b"tag1" + salt + cipher_digits.encode('utf-8'), hashlib.sha256).hexdigest()
    tag2 = hmac.new(sk2, b"tag2" + salt + cipher_digits.encode('utf-8'), hashlib.sha256).hexdigest()
    tag3 = hmac.new(sk3, b"tag3" + salt + cipher_digits.encode('utf-8'), hashlib.sha256).hexdigest()
    tag4 = hmac.new(sk4, b"tag4" + salt + cipher_digits.encode('utf-8'), hashlib.sha256).hexdigest()

    return {
        "salt": salt.hex(),
        "cipher": cipher_digits,
        "tags": {"t1": tag1, "t2": tag2, "t3": tag3, "t4": tag4},
        "meta": {"fluid_digits": FLUID_NUM_DIGITS, "div_digits": DIVISION_DIGITS, "charset_len": len(CHARSET)}
    }

def decrypt_numeric(password: str, envelope: Dict[str, Any]) -> str:
    salt = bytes.fromhex(envelope["salt"])
    cipher_digits = envelope["cipher"]
    tags = envelope["tags"]
    meta = envelope["meta"]

    master = derive_master_key(password, salt)
    sk1 = derive_subkey(master, b"stage1-fluid")
    sk2 = derive_subkey(master, b"stage2-div")
    sk3 = derive_subkey(master, b"stage3-keyop")
    sk4 = derive_subkey(master, b"stage4-xor")

    # Verify tags in order (stage gating)
    expected_t1 = hmac.new(sk1, b"tag1" + salt + cipher_digits.encode('utf-8'), hashlib.sha256).hexdigest()
    if not hmac.compare_digest(expected_t1, tags["t1"]):
        raise ValueError("Stage1 verification failed (invalid password or tampered).")
    expected_t2 = hmac.new(sk2, b"tag2" + salt + cipher_digits.encode('utf-8'), hashlib.sha256).hexdigest()
    if not hmac.compare_digest(expected_t2, tags["t2"]):
        raise ValueError("Stage2 verification failed.")
    expected_t3 = hmac.new(sk3, b"tag3" + salt + cipher_digits.encode('utf-8'), hashlib.sha256).hexdigest()
    if not hmac.compare_digest(expected_t3, tags["t3"]):
        raise ValueError("Stage3 verification failed.")
    expected_t4 = hmac.new(sk4, b"tag4" + salt + cipher_digits.encode('utf-8'), hashlib.sha256).hexdigest()
    if not hmac.compare_digest(expected_t4, tags["t4"]):
        raise ValueError("Stage4 verification failed.")

    # Stage 4 reverse: numeric string -> bytes, XOR with sk4 to get stage3_all_bytes
    cipher_bytes = digits_to_bytes(cipher_digits)
    ks_full = gen_xor_mask_bytes(sk4, len(cipher_bytes))
    stage3_all_bytes = xor_bytes(cipher_bytes, ks_full)

    # Stage 3 reverse: split into per-chunk bytes and XOR with per-chunk ks to recover stage2_bytes
    chunk_bytes = DIVISION_DIGITS // 3
    chunks = [stage3_all_bytes[i:i+chunk_bytes] for i in range(0, len(stage3_all_bytes), chunk_bytes)]
    recovered_stage2_bytes_parts = []
    for idx, token in enumerate(chunks):
        ks = gen_xor_mask_bytes(hmac.new(sk3, b'chunkidx_' + idx.to_bytes(4,'big'), hashlib.sha256).digest(), len(token))
        orig_chunk = xor_bytes(token, ks)
        recovered_stage2_bytes_parts.append(orig_chunk)
    recovered_stage2_bytes = b''.join(recovered_stage2_bytes_parts)

    # Stage 2 reverse: bytes -> decimal string (3-digit per byte)
    recovered_stage2_digits = bytes_to_digits(recovered_stage2_bytes)

    # invert Number_div mapping: group recovered_stage2_digits into div_digits groups and map back
    Number_div = gen_number_division_table(sk2, digits=meta["div_digits"])
    inv_Number_div = {v: k for k, v in Number_div.items()}

    if len(recovered_stage2_digits) % meta["div_digits"] != 0:
        raise ValueError("Recovered stage2 length mismatch.")

    digits_reconstructed = []
    for i in range(0, len(recovered_stage2_digits), meta["div_digits"]):
        piece = recovered_stage2_digits[i:i+meta["div_digits"]]
        d = inv_Number_div.get(piece)
        if d is None:
            raise ValueError(f"Failed to invert Number_div mapping during decryption. Piece: {piece}")
        digits_reconstructed.append(d)
    concat = "".join(digits_reconstructed)

    # Stage 1 reverse: split concat into fluid_digits groups and invert Fluid_table
    Fluid_table = gen_fluid_table(sk1, charset=CHARSET, digits=meta["fluid_digits"])
    inv_fluid = {v: k for k, v in Fluid_table.items()}

    if len(concat) % meta["fluid_digits"] != 0:
        raise ValueError("Recovered concat length mismatch.")

    recovered_chars = []
    for i in range(0, len(concat), meta["fluid_digits"]):
        piece = concat[i:i+meta["fluid_digits"]]
        ch = inv_fluid.get(piece)
        if ch is None:
            raise ValueError(f"Failed to map fluid piece back to character. Piece: {piece}")
        recovered_chars.append(ch)

    return "".join(recovered_chars)

# ---------------- Example usage ----------------
if __name__ == "__main__":
    pw = "gaurav1324@@**"
    pt = "helloWorld123!"
    env = encrypt_numeric(pw, pt)
    print("Envelope meta/tags:", {k: v for k, v in env.items() if k != 'cipher'})
    rec = decrypt_numeric(pw, env)
    print("Recovered:", rec)
    assert rec == pt, "Decryption failed to recover plaintext"
    print("Success: plaintext recovered")


Envelope meta/tags: {'salt': 'e8a4bbe9440aa04dd2cf820aa76a480b', 'tags': {'t1': 'f197694c8bd01ac55c44647aacc8b483ff0d72a4c9628120290e7d3cdeb5fc0a', 't2': '9bc60ba3e932eb93972c7116df3bbb152e0bf5b43f39bffd0c3e1798019329ad', 't3': '161239a3098babd16d21aaa01b498d73e9b5f08c9c546b56a176b5436a2baec5', 't4': '832b2b8b9d5fb0357ce2417ef28d789ba3b267e76f815ad4ece4e180459e6ed4'}, 'meta': {'fluid_digits': 12, 'div_digits': 3, 'charset_len': 95}}
Recovered: helloWorld123!
Success: plaintext recovered


In [1]:
# --- Section 1: Imports & Configuration ---

import os
import hmac
import hashlib
from typing import Dict, Any

# Configuration constants
PBKDF2_ITERS = 200_000
MASTER_KEY_LEN = 32            # 256-bit master key
FLUID_NUM_DIGITS = 12          # must be multiple of 3
DIVISION_DIGITS = 3            # must be 3 (1 byte per division code)
KEYOP_DIGITS = 6
CHARSET = [chr(i) for i in range(32, 127)]  # printable ASCII characters


🧠 Section 2 — Key Derivation Functions

In [ ]:
# --- Section 2: KDF and DRBG helpers ---

def derive_master_key(password: str, salt: bytes) -> bytes:
    """Derive a fixed-length master key from password+salt using PBKDF2-HMAC-SHA256."""
    print("Derive master Key",hashlib.pbkdf2_hmac('sha256', password.encode('utf-8'), salt, PBKDF2_ITERS, dklen=MASTER_KEY_LEN))
    return hashlib.pbkdf2_hmac('sha256', password.encode('utf-8'), salt, PBKDF2_ITERS, dklen=MASTER_KEY_LEN)

def derive_subkey(master_key: bytes, info: bytes) -> bytes:
    """Derive deterministic subkey using HMAC(master_key, info)."""
    print("Derive subkey", hmac.new(master_key, info, hashlib.sha256).digest())
    return hmac.new(master_key, info, hashlib.sha256).digest()

def drbg_bytes(subkey: bytes, label: bytes, counter: int, out_bytes: int) -> bytes:
    """Deterministic pseudo-random bytes using HMAC chaining."""
    out = bytearray()
    ctr = counter
    while len(out) < out_bytes:
        out.extend(hmac.new(subkey, label + ctr.to_bytes(4, 'big'), hashlib.sha256).digest())
        ctr += 1
    print("drbg_bytes", bytes(out[:out_bytes]))
    return bytes(out[:out_bytes])




⚙️ Section 3 — Table Generators

In [4]:
# --- Section 3: Fluid and Number Division Table Generators ---

def gen_fluid_table(subkey: bytes, charset=CHARSET, digits=FLUID_NUM_DIGITS) -> Dict[str, str]:
    """
    Deterministic Fluid table: char -> numeric string of length `digits`.
    Each char maps to `digits/3` groups of 3-digit values (000–255).
    """
    if digits % 3 != 0:
        raise ValueError("FLUID_NUM_DIGITS must be a multiple of 3.")
    groups = digits // 3
    table = {}
    for i, ch in enumerate(charset):
        parts = []
        for g in range(groups):
            raw = drbg_bytes(subkey, b'fluid_' + bytes([i % 256]) + bytes([g % 256]), counter=g, out_bytes=2)
            val = int.from_bytes(raw, 'big') % 256
            parts.append(f"{val:03d}")
        table[ch] = "".join(parts)
    print("gen_fluid_table", table)
    return table

def gen_number_division_table(subkey: bytes, digits=DIVISION_DIGITS) -> Dict[str, str]:
    """Digit (0–9) → 3-digit string (000–255)."""
    if digits != 3:
        raise ValueError("DIVISION_DIGITS must be 3 in this implementation.")
    table = {}
    for d in range(10):
        raw = drbg_bytes(subkey, b'div_' + bytes([d]), counter=d, out_bytes=2)
        val = int.from_bytes(raw, 'big') % 256
        table[str(d)] = f"{val:03d}"
    print("gen_number_division_table", table)
    return table


🔐 Section 4 — XOR and Conversion Helpers

In [5]:
# --- Section 4: Utility Functions ---

def gen_xor_mask_bytes(subkey: bytes, length_bytes: int) -> bytes:
    """Generate deterministic XOR keystream from a subkey."""
    out = bytearray()
    counter = 0
    while len(out) < length_bytes:
        blk = hmac.new(subkey, b'xor_bytes_' + counter.to_bytes(4, 'big'), hashlib.sha256).digest()
        out.extend(blk)
        counter += 1
    print("gen_xor_mask_bytes", bytes(out[:length_bytes]))
    return bytes(out[:length_bytes])

def xor_bytes(a: bytes, b: bytes) -> bytes:
    """XOR two byte arrays."""
    print("xor_bytes", bytes(x ^ y for x, y in zip(a, b)))
    return bytes(x ^ y for x, y in zip(a, b))

def digits_to_bytes(numeric_string: str) -> bytes:
    """Convert decimal string (000–255 per 3 digits) → bytes."""
    assert len(numeric_string) % 3 == 0
    print("digits_to_bytes", bytes(int(numeric_string[i:i+3]) for i in range(0, len(numeric_string), 3)))
    return bytes(int(numeric_string[i:i+3]) for i in range(0, len(numeric_string), 3))

def bytes_to_digits(b: bytes) -> str:
    """Convert bytes → 3-digit decimal string per byte."""
    print("bytes_to_digits", ''.join(f'{x:03d}' for x in b))
    return ''.join(f'{x:03d}' for x in b)


🧩 Section 5 — Encryption Function

In [6]:
# --- Section 5: Encryption ---

def encrypt_numeric(password: str, plaintext: str) -> Dict[str, Any]:
    salt = os.urandom(16)
    master = derive_master_key(password, salt)

    # Stage subkeys
    sk1 = derive_subkey(master, b"stage1-fluid")
    sk2 = derive_subkey(master, b"stage2-div")
    sk3 = derive_subkey(master, b"stage3-keyop")
    sk4 = derive_subkey(master, b"stage4-xor")
    print("Subkeys derived", sk1, sk2, sk3, sk4)

    # Generate tables
    Fluid_table = gen_fluid_table(sk1, charset=CHARSET)
    Number_div = gen_number_division_table(sk2)
    print("Tables generated", Fluid_table, Number_div)
    
    # Stage 1: char → numeric (fluid)
    concat = "".join(Fluid_table[ch] for ch in plaintext)

    # Stage 2: each digit → 3-digit mapping
    stage2 = "".join(Number_div[d] for d in concat)
    stage2_bytes = digits_to_bytes(stage2)

    print("Stage 2 bytes", stage2_bytes)
    # Stage 3: chunk-wise XOR with sk3
    chunks = [stage2_bytes[i:i+1] for i in range(len(stage2_bytes))]
    stage3_all = b''.join(xor_bytes(
        c,
        gen_xor_mask_bytes(hmac.new(sk3, b'chunkidx_' + i.to_bytes(4,'big'), hashlib.sha256).digest(), len(c))
    ) for i, c in enumerate(chunks))

    # Stage 4: XOR whole stream with sk4
    ks_full = gen_xor_mask_bytes(sk4, len(stage3_all))
    cipher_bytes = xor_bytes(stage3_all, ks_full)
    cipher_digits = bytes_to_digits(cipher_bytes)

    # Verification tags
    tag1 = hmac.new(sk1, b"tag1" + salt + cipher_digits.encode(), hashlib.sha256).hexdigest()
    tag2 = hmac.new(sk2, b"tag2" + salt + cipher_digits.encode(), hashlib.sha256).hexdigest()
    tag3 = hmac.new(sk3, b"tag3" + salt + cipher_digits.encode(), hashlib.sha256).hexdigest()
    tag4 = hmac.new(sk4, b"tag4" + salt + cipher_digits.encode(), hashlib.sha256).hexdigest()
    print("Verification tags", tag1, tag2, tag3, tag4)
    return {
        "salt": salt.hex(),
        "cipher": cipher_digits,
        "tags": {"t1": tag1, "t2": tag2, "t3": tag3, "t4": tag4},
        "meta": {"fluid_digits": FLUID_NUM_DIGITS, "div_digits": DIVISION_DIGITS}
    }


🔓 Section 6 — Decryption Function

In [7]:
# --- Section 6: Decryption ---

def decrypt_numeric(password: str, envelope: Dict[str, Any]) -> str:
    salt = bytes.fromhex(envelope["salt"])
    cipher_digits = envelope["cipher"]
    tags = envelope["tags"]
    meta = envelope["meta"]
    print("Decryption envelope", salt, cipher_digits, tags, meta)

    master = derive_master_key(password, salt)
    sk1 = derive_subkey(master, b"stage1-fluid")
    sk2 = derive_subkey(master, b"stage2-div")
    sk3 = derive_subkey(master, b"stage3-keyop")
    sk4 = derive_subkey(master, b"stage4-xor")

    print("Subkeys derived", sk1, sk2, sk3, sk4)

    # Verify tags
    for stage, subkey, tagname in [(1, sk1, "t1"), (2, sk2, "t2"), (3, sk3, "t3"), (4, sk4, "t4")]:
        expected = hmac.new(subkey, f"tag{stage}".encode() + salt + cipher_digits.encode(), hashlib.sha256).hexdigest()
        if not hmac.compare_digest(expected, tags[tagname]):
            raise ValueError(f"Stage {stage} verification failed.")

    # Stage 4 reverse
    cipher_bytes = digits_to_bytes(cipher_digits)
    ks_full = gen_xor_mask_bytes(sk4, len(cipher_bytes))
    stage3_all = xor_bytes(cipher_bytes, ks_full)
    print("Stage 3 all bytes", stage3_all)

    # Stage 3 reverse
    chunks = [stage3_all[i:i+1] for i in range(len(stage3_all))]
    recovered_stage2_bytes = b''.join(
        xor_bytes(
            token,
            gen_xor_mask_bytes(hmac.new(sk3, b'chunkidx_' + i.to_bytes(4,'big'), hashlib.sha256).digest(), len(token))
        )
        for i, token in enumerate(chunks)
    )
    print("Stage 2 recovered bytes", recovered_stage2_bytes)
    recovered_stage2_digits = bytes_to_digits(recovered_stage2_bytes)

    # Stage 2 reverse
    Number_div = gen_number_division_table(sk2, digits=meta["div_digits"])
    inv_Number_div = {v: k for k, v in Number_div.items()}

    print("inv_Number_div", inv_Number_div)

    digits_reconstructed = []
    for i in range(0, len(recovered_stage2_digits), meta["div_digits"]):
        piece = recovered_stage2_digits[i:i+meta["div_digits"]]
        if piece not in inv_Number_div:
            raise ValueError(f"Cannot invert division piece: {piece}")
        digits_reconstructed.append(inv_Number_div[piece])
    print("digits_reconstructed", digits_reconstructed)
    concat = "".join(digits_reconstructed)

    # Stage 1 reverse
    Fluid_table = gen_fluid_table(sk1, charset=CHARSET, digits=meta["fluid_digits"])
    inv_fluid = {v: k for k, v in Fluid_table.items()}
    print("inv_fluid", inv_fluid)

    recovered_chars = []
    for i in range(0, len(concat), meta["fluid_digits"]):
        piece = concat[i:i+meta["fluid_digits"]]
        if piece not in inv_fluid:
            raise ValueError(f"Cannot invert fluid piece: {piece}")
        recovered_chars.append(inv_fluid[piece])
    print("recovered_chars", recovered_chars)

    return "".join(recovered_chars)


🧪 Section 7 — Example Usage

In [8]:
# --- Section 7: Example Usage ---

pw = "gaurav1324@@**"
pt = "shakti!"

env = encrypt_numeric(pw, pt)
print("Encrypted Envelope:")
print(env)

rec = decrypt_numeric(pw, env)
print("\nRecovered Plaintext:", rec)
assert rec == pt, "Decryption failed!"
print("\n✅ Success: plaintext recovered correctly.")


Derive master Key b'N\xeb\xbbv\x85:\n\xe3SHP\xa2\xc9\x02\x9cIi\xf7\xb4*~\xf8D\xd1\xbb\x01Q\x96^\xe2\x92A'
Subkeys derived b',\x91QY$C\xb5\xdf3\xf59\x83\xee\x8f\xc2e\xbc\\\'"n\x8aKi\xc5\x97\xb5\xcfx\xcfq\x93' b"\xaa\xf8\xe2h`_\xec\x8d\x87\x0c\xcb\x86\x98'\t\xb7\xde!\x91\xc8\x0f\xb7J\x11\xc9\x10\x81\xb3\xc0M\x11\xa1" b'\xe7sQh{\x8a\xd4\x01\x98\xd99\xf08\xccO\x85\x90\xbb\xf8\xe2\xdb\x8f\x13M\x006&\xaf\xd2o \x1c' b'\xfd\xa2\xaf\\E\xc7>\x0f\x9e\xc5\xb7\xda\t\x9b\xe4~\xad>V\x92:\xf8\xc0-\xc78\x12\xb0\x9a\xdf\x08\xe0'
gen_fluid_table {' ': '151098123219', '!': '063171064036', '"': '199071153234', '#': '196207065139', '$': '085164178183', '%': '028107132133', '&': '031213092102', "'": '190237040146', '(': '204120026047', ')': '155210200025', '*': '082144094004', '+': '073149102188', ',': '084124012040', '-': '043061022141', '.': '234145052144', '/': '172042239236', '0': '081246075114', '1': '101065197020', '2': '005218172196', '3': '146174202103', '4': '070223223173', '5': '178240145141', '6':